# Simulating Customer Response to a Pricing Change

How do different customer segments react when a SaaS product raises its price?

This notebook demonstrates how to use TinyTroupe to simulate the responses of four distinct customer personas to a 50% price increase — from **\$12/month to \$18/month** — for a fictional productivity SaaS product called **FlowDesk**.

**What you will learn:**
- How to define reusable personas as JSON files and load them with `TinyPerson.load_specification()`
- How to deliver a scenario to all agents simultaneously using `TinyWorld.broadcast()`
- How to extract structured, multi-field insights using `ResultsExtractor`
- How to analyze churn risk and feature requests across customer segments

**Scientific framing:** Price sensitivity is shaped by reference price anchoring (Kahneman & Tversky, 1979), perceived value relative to alternatives (Monroe, 1990), and loss aversion — a 50% increase is framed as a significant loss, and losses loom larger than equivalent gains in human decision-making. Expect asymmetric reactions across personas based on income elasticity and switching cost.

**Prerequisites:** This notebook requires an OpenAI API key (or Azure OpenAI credentials) configured in `config.ini`. See the main [TinyTroupe README](../README.md) for setup instructions. All product names and personas in this example are entirely fictional.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.validation import TinyPersonValidator
from tinytroupe.extraction import ResultsExtractor
import tinytroupe.control as control

## Product & Scenario

**Product:** FlowDesk — a fictional productivity SaaS combining task management, time tracking, and lightweight project collaboration.

**Pricing change:** Monthly subscription increases from **\$12/month to \$18/month** (a 50% increase), announced by email with one billing cycle of notice.

We simulate four customer types who differ on income sensitivity, switching cost, usage intensity, and trust in the vendor.

## Customer Personas

Four personas designed to capture meaningfully different segments of a productivity SaaS customer base. Each is defined in a JSON file under `./agents/` and loaded with `TinyPerson.load_specification()` — the same pattern used by the built-in Oscar and Lisa agents.

| Persona | Age | Role | Monthly budget pressure | Switching cost |
|---------|-----|------|------------------------|----------------|
| Maya Chen | 27 | Solo freelancer | Very high (variable income) | Low |
| Raj Patel | 34 | Startup founder | High (limited runway) | Medium |
| Sarah Kim | 41 | Enterprise PM | Low (expense account) | High |
| Ethan Brooks | 22 | Grad student | Very high (student income) | Very low |

In [ ]:
# Load each persona from its JSON specification in ./agents/
# This is the same pattern as create_oscar_the_architect() in tinytroupe/examples/agents.py
maya  = TinyPerson.load_specification('./agents/MayaChen.agent.json')
raj   = TinyPerson.load_specification('./agents/RajPatel.agent.json')
sarah = TinyPerson.load_specification('./agents/SarahKim.agent.json')
ethan = TinyPerson.load_specification('./agents/EthanBrooks.agent.json')

all_personas = [maya, raj, sarah, ethan]

In [ ]:
# Inspect each persona to confirm they loaded correctly before running the simulation
for persona in all_personas:
    persona.minibio()

## Simulation Setup

We place all four personas in a shared `TinyWorld` with `broadcast_if_no_target=False`. This means each agent receives the same scenario and responds independently — we are **not** simulating a group discussion, but four parallel individual reactions to the same announcement. This is the correct setup for individual customer research.

Simulation caching via `control.begin()` avoids redundant API calls on repeated runs. The `.cache.` naming convention marks the cache file as non-committable (see `.gitignore`).

In [ ]:
# Uncomment to cache the simulation and avoid rerunning expensive API calls
# control.begin("pricing_change_simulation.cache.json")

# broadcast_if_no_target=False: each agent reacts independently, not as a group
world = TinyWorld('FlowDesk Customer Panel', all_personas, broadcast_if_no_target=False)

## Scenario Delivery

We deliver the scenario in three parts:
1. **Product context** — orient each persona as a paying FlowDesk customer
2. **Pricing announcement** — the actual email they received
3. **Reflection primer** — an inner monologue broadcast via `broadcast_thought()` that encourages honest, in-character reasoning rather than a surface reaction

Then we run one simulation step, which causes each agent to act based on everything they have heard and thought.

In [ ]:
product_context = (
    'You are a paying customer of FlowDesk, a productivity SaaS tool you have been using '
    'for the past year. FlowDesk helps you manage tasks, track time, and collaborate on '
    'projects. You currently pay $12/month and have integrated it into your daily workflow.'
)

pricing_announcement = (
    'You just received this email from FlowDesk:\n\n'
    'Subject: Important update to your FlowDesk subscription\n\n'
    'Hi there,\n\n'
    'We are writing to let you know that starting next billing cycle, your FlowDesk '
    'monthly subscription will increase from $12/month to $18/month. This change reflects '
    'our ongoing investment in new collaboration features, a redesigned mobile app, and '
    'faster customer support.\n\n'
    'Your account, data, and current plan features remain unchanged. If you have '
    'questions, our support team is here to help.\n\n'
    'Thank you for being a FlowDesk customer.\n'
    '\u2014 The FlowDesk Team\n\n'
    'How do you feel about this? What will you do?'
)

# Inject an inner monologue to encourage honest, nuanced reflection rather than a
# surface reaction. broadcast_thought() delivers this as an internal thought, not
# as a message from another agent.
reflection_primer = (
    'I will think carefully and honestly about this situation. I will consider: '
    'how much I actually use and value FlowDesk, how $6/month more fits into my budget, '
    'whether I trust this company and its communication, whether I would look for '
    'alternatives, and what specific improvements I would need to see to feel the '
    'price increase is justified. I will not give a polite or generic answer '
    '\u2014 I will respond as I genuinely would in real life.'
)

world.broadcast(product_context)
world.broadcast(pricing_announcement)
world.broadcast_thought(reflection_primer)
world.run(1)

In [ ]:
# Preview each agent's raw response before structured extraction.
# This is useful for understanding what the LLM produced and for debugging
# extraction issues when field values come back as N/A.
for persona in all_personas:
    print(persona.pretty_current_interactions(max_content_length=500))
    print()

## Persona Validation (Optional)

We can spot-check Maya — a price-sensitive persona — using `TinyPersonValidator`. This iteratively questions the agent and returns a confidence score (0–1) indicating how well the agent's behavior matches our expectations. A high score confirms the persona definition is producing coherent, realistic responses.

This step is disabled by default because it makes additional API calls. Set `RUN_PERSONA_VALIDATION = True` in the next cell if you want to run it.

In [ ]:
maya_expectations = """
Maya is:
- Budget-sensitive and cost-conscious — tracks every subscription expense
- A freelancer with variable monthly income around $4,000-$5,000
- Aware of free and cheaper alternatives (Notion, Trello, Toggl)
- Not emotionally attached to FlowDesk -- pragmatic about switching

Expected behavior on a 50% price increase:
- Expresses frustration or concern about the cost increase
- Considers or explicitly mentions cancellation or switching to an alternative
- Requests a justification, discount, or annual billing option to stay
- Is unlikely to simply accept the price increase without any pushback
"""

RUN_PERSONA_VALIDATION = False

if RUN_PERSONA_VALIDATION:
    maya_score, maya_justification = TinyPersonValidator.validate_person(
        maya,
        expectations=maya_expectations,
        include_agent_spec=True,
        max_content_length=None
    )

    print(f"Maya validation score: {maya_score:.2f} / 1.0")
    print(f"Justification: {maya_justification}")
else:
    print("Persona validation skipped. Set RUN_PERSONA_VALIDATION = True to run it.")

## Extract Results

`ResultsExtractor` uses an LLM to parse each agent's interaction history and pull structured data into the fields we define. Precise `fields_hints` are critical: they constrain the LLM to produce consistently formatted values (e.g., integer 1–5 scales, fixed categorical labels) that can be reliably compared across personas.

In [ ]:
results_extractor = ResultsExtractor(
    extraction_objective=(
        'Determine how the customer persona responded to a 50% price increase '
        '($12/month to $18/month) for a productivity SaaS called FlowDesk. '
        'Extract their switching risk, likelihood to continue, likelihood to churn, '
        'likelihood to recommend, any specific product improvements they requested, '
        'and their overall reaction.'
    ),
    situation=(
        'The agent received an email announcing a price increase from $12 to $18/month '
        'and was asked to respond honestly and in character.'
    ),
    fields=[
        'name',
        'switching_risk',
        'likelihood_to_continue',
        'likelihood_to_churn',
        'likelihood_to_recommend',
        'requested_improvements',
        'overall_reaction',
    ],
    fields_hints={
        'switching_risk': (
            'One of: Very High, High, Medium, Low, Very Low. '
            'Very High means almost certain to leave; Very Low means extremely unlikely.'
        ),
        'likelihood_to_continue': (
            'Integer 1-5. 1 = will definitely cancel, 5 = will definitely keep paying. '
            'Use N/A if truly indeterminate.'
        ),
        'likelihood_to_churn': (
            'Integer 1-5. 1 = will definitely stay, 5 = will definitely cancel. '
            'Use N/A if truly indeterminate.'
        ),
        'likelihood_to_recommend': (
            'Integer 1-5. 1 = would never recommend, 5 = would strongly recommend. '
            'Use N/A if truly indeterminate.'
        ),
        'requested_improvements': (
            'Comma-separated list of specific product changes the persona mentioned. '
            'Write "None mentioned" if no specific improvements were requested.'
        ),
        'overall_reaction': (
            'One of: Strongly Positive, Positive, Neutral, Negative, Strongly Negative.'
        ),
    },
    verbose=True,
)

In [ ]:
results = results_extractor.extract_results_from_agents(world.agents)

# Filter to valid dict results only
filtered_results = [r for r in results if isinstance(r, dict)]

df = pd.DataFrame(filtered_results)
df

## Analysis

We visualize three key behavioral intent scores across all four customer segments, then print a structured per-persona summary of all extracted dimensions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Colors encode expected churn risk consistently across all three charts:
# red = high risk (Maya, Ethan), orange = medium (Raj), blue = low (Sarah)
persona_colors = ['#e74c3c', '#e67e22', '#3498db', '#e74c3c']

# Churn likelihood by persona
churn_data = df[["name", "likelihood_to_churn"]].copy()
churn_data["likelihood_to_churn"] = pd.to_numeric(
    churn_data["likelihood_to_churn"].replace("N/A", None), errors="coerce"
)
churn_data = churn_data.dropna()
axes[0].bar(churn_data["name"], churn_data["likelihood_to_churn"],
            color=persona_colors[:len(churn_data)])
axes[0].set_title("Churn Likelihood\n(1 = Stay, 5 = Leave)")
axes[0].set_ylabel("Score (1-5)")
axes[0].set_ylim(0, 5.5)
axes[0].tick_params(axis="x", rotation=20)

# Likelihood to continue
continue_data = df[["name", "likelihood_to_continue"]].copy()
continue_data["likelihood_to_continue"] = pd.to_numeric(
    continue_data["likelihood_to_continue"].replace("N/A", None), errors="coerce"
)
continue_data = continue_data.dropna()
axes[1].bar(continue_data["name"], continue_data["likelihood_to_continue"],
            color=persona_colors[:len(continue_data)])
axes[1].set_title("Likelihood to Continue\n(1 = Cancel, 5 = Stay)")
axes[1].set_ylabel("Score (1-5)")
axes[1].set_ylim(0, 5.5)
axes[1].tick_params(axis="x", rotation=20)

# Likelihood to recommend
recommend_data = df[["name", "likelihood_to_recommend"]].copy()
recommend_data["likelihood_to_recommend"] = pd.to_numeric(
    recommend_data["likelihood_to_recommend"].replace("N/A", None), errors="coerce"
)
recommend_data = recommend_data.dropna()
axes[2].bar(recommend_data["name"], recommend_data["likelihood_to_recommend"],
            color=persona_colors[:len(recommend_data)])
axes[2].set_title("Likelihood to Recommend\n(1 = Never, 5 = Strongly)")
axes[2].set_ylabel("Score (1-5)")
axes[2].set_ylim(0, 5.5)
axes[2].tick_params(axis="x", rotation=20)

plt.suptitle(
    "FlowDesk Price Increase: Customer Response by Persona",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 70)
print("FLOWDESK PRICING CHANGE -- PERSONA RESPONSE SUMMARY")
print("=" * 70)

for _, row in df.iterrows():
    print(f'\n{row.get("name", "Unknown")}')
    print(f'  Switching risk     : {row.get("switching_risk", "N/A")}')
    print(f'  Likelihood to stay : {row.get("likelihood_to_continue", "N/A")} / 5')
    print(f'  Likelihood to churn: {row.get("likelihood_to_churn", "N/A")} / 5')
    print(f'  Likelihood to rec. : {row.get("likelihood_to_recommend", "N/A")} / 5')
    print(f'  Overall reaction   : {row.get("overall_reaction", "N/A")}')
    print(f'  Requested changes  : {row.get("requested_improvements", "N/A")}')

print("\n" + "=" * 70)

In [ ]:
# control.end()